In [2]:
piece_values = {
    'P': 1, 'N': 3, 'B': 3, 'R': 5, 'Q': 9, 'K': 0,
    'p': -1, 'n': -3, 'b': -3, 'r': -5, 'q': -9, 'k': 0
}

def evaluate(board):
    return sum(piece_values.get(cell, 0) for row in board for cell in row)

def generate_moves(board, white_turn):
    moves = []
    directions = {
        'P': [(-1, 0)], 'p': [(1, 0)],
        'N': [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)],
        'n': [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]
    }
    for i in range(8):
        for j in range(8):
            piece = board[i][j]
            if piece == '.':
                continue
            if white_turn and piece.islower():
                continue
            if not white_turn and piece.isupper():
                continue
            if piece in directions:
                for di, dj in directions[piece]:
                    ni, nj = i + di, j + dj
                    if 0 <= ni < 8 and 0 <= nj < 8:
                        target = board[ni][nj]
                        if target == '.' or (target.islower() if white_turn else target.isupper()):
                            new_board = [row[:] for row in board]
                            new_board[ni][nj] = piece
                            new_board[i][j] = '.'
                            moves.append(new_board)
    return moves

def beam_search(board, beam_width, depth, white_turn=True):
    beam = [(board, [], evaluate(board))]
    for _ in range(depth):
        candidates = []
        for b, seq, score in beam:
            for new_b in generate_moves(b, white_turn):
                new_seq = seq + [new_b]
                new_score = evaluate(new_b)
                candidates.append((new_b, new_seq, new_score))
        candidates.sort(key=lambda x: x[2], reverse=white_turn)
        beam = candidates[:beam_width]
        white_turn = not white_turn
        if not beam:
            break
    best = max(beam, key=lambda x: x[2])
    return best[1], best[2]

board = [
    ['r','n','b','q','k','b','n','r'],
    ['p','p','p','p','p','p','p','p'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['P','P','P','P','P','P','P','P'],
    ['R','N','B','Q','K','B','N','R']
]

moves, score = beam_search(board, 3, 3)
print(score)
print(len(moves))

0
3


In [4]:
import math
import random

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def total_distance(route):
    dist = 0
    for i in range(len(route)):
        dist += distance(route[i], route[(i + 1) % len(route)])
    return dist

def get_neighbor(route):
    a, b = random.sample(range(len(route)), 2)
    new_route = route[:]
    new_route[a], new_route[b] = new_route[b], new_route[a]
    return new_route

def hill_climbing(points, max_iterations=1000):
    current_route = points[:]
    random.shuffle(current_route)
    current_distance = total_distance(current_route)

    for _ in range(max_iterations):
        neighbor = get_neighbor(current_route)
        neighbor_distance = total_distance(neighbor)
        if neighbor_distance < current_distance:
            current_route = neighbor
            current_distance = neighbor_distance

    return current_route, current_distance

points = [(0,0), (2,3), (5,4), (1,1), (6,2), (3,7)]

route, dist = hill_climbing(points)

print(route)
print(dist)

[(6, 2), (5, 4), (3, 7), (2, 3), (1, 1), (0, 0)]
19.939561738791085


In [5]:
import random
import math

cities = [(0,0),(2,3),(5,4),(1,1),(6,2),(3,7),(8,3),(7,6),(4,5),(9,1)]

population_size = 50
max_generations = 100
mutation_rate = 0.1

def distance(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def total_distance(route):
    dist = 0
    for i in range(len(route)):
        dist += distance(route[i], route[(i+1)%len(route)])
    return dist

def evaluate_fitness(route):
    return total_distance(route)

def create_random_route():
    route = cities[:]
    random.shuffle(route)
    return route

def select_parents(population, fitness_scores):
    sorted_population = [x for _, x in sorted(zip(fitness_scores, population))]
    return sorted_population[:len(population)//2]

def crossover(parent1, parent2):
    start, end = sorted(random.sample(range(len(parent1)), 2))
    child = [None]*len(parent1)
    child[start:end] = parent1[start:end]
    ptr = 0
    for city in parent2:
        if city not in child:
            while child[ptr] is not None:
                ptr += 1
            child[ptr] = city
    return child

def mutate(route):
    a, b = random.sample(range(len(route)), 2)
    route[a], route[b] = route[b], route[a]
    return route

population = [create_random_route() for _ in range(population_size)]

for generation in range(max_generations):
    fitness_scores = [evaluate_fitness(route) for route in population]
    best_fitness = min(fitness_scores)
    print("Generation", generation+1, "Best Distance:", best_fitness)
    parents = select_parents(population, fitness_scores)
    new_population = []
    while len(new_population) < population_size:
        parent1, parent2 = random.sample(parents, 2)
        child = crossover(parent1, parent2)
        if random.random() < mutation_rate:
            child = mutate(child)
        new_population.append(child)
    population = new_population

fitness_scores = [evaluate_fitness(route) for route in population]
best_route = population[fitness_scores.index(min(fitness_scores))]

print("\nBest Route:")
print(best_route)
print("Total Distance:", min(fitness_scores))

Generation 1 Best Distance: 38.60234287477388
Generation 2 Best Distance: 34.171113100970125
Generation 3 Best Distance: 34.171113100970125
Generation 4 Best Distance: 33.78570472528004
Generation 5 Best Distance: 34.92889004033229
Generation 6 Best Distance: 32.80260714631797
Generation 7 Best Distance: 34.50594097970435
Generation 8 Best Distance: 30.337907376459253
Generation 9 Best Distance: 30.337907376459253
Generation 10 Best Distance: 30.337907376459253
Generation 11 Best Distance: 30.337907376459253
Generation 12 Best Distance: 30.337907376459253
Generation 13 Best Distance: 30.004056841037066
Generation 14 Best Distance: 31.773558391079376
Generation 15 Best Distance: 31.773558391079376
Generation 16 Best Distance: 31.796851203241246
Generation 17 Best Distance: 30.50675774624713
Generation 18 Best Distance: 30.50675774624713
Generation 19 Best Distance: 29.13727444828293
Generation 20 Best Distance: 29.13727444828293
Generation 21 Best Distance: 29.13727444828293
Generation 

In [6]:
import random

def evaluate(processors):
    return max(sum(task[1] for task in p) for p in processors)

def beam_search(tasks, num_processors, beam_width, depth):
    initial = [[] for _ in range(num_processors)]
    beam = [(initial, 0, 0)]

    for _ in range(depth):
        candidates = []
        for alloc, index, _ in beam:
            if index >= len(tasks):
                candidates.append((alloc, index, evaluate(alloc)))
                continue
            task = tasks[index]
            for i in range(num_processors):
                new_alloc = [p[:] for p in alloc]
                new_alloc[i].append(task)
                score = evaluate(new_alloc)
                candidates.append((new_alloc, index + 1, score))
        candidates.sort(key=lambda x: x[2])
        beam = candidates[:beam_width]

    best = min(beam, key=lambda x: x[2])
    return best[0], best[2]

tasks = [
    ("T1", 4, 2),
    ("T2", 2, 1),
    ("T3", 6, 3),
    ("T4", 3, 2),
    ("T5", 5, 1)
]

num_processors = 3

allocation, load = beam_search(tasks, num_processors, beam_width=3, depth=len(tasks))

print("Allocation:")
for i, p in enumerate(allocation):
    print("Processor", i+1, ":", p)

print("Max Load:", load)

Allocation:
Processor 1 : [('T1', 4, 2), ('T5', 5, 1)]
Processor 2 : [('T2', 2, 1), ('T4', 3, 2)]
Processor 3 : [('T3', 6, 3)]
Max Load: 9
